In [1]:
import os
import json
import ast
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
import subprocess

# Data Path

In [3]:
root_data_path="taskB"

In [4]:
#Training data path
taskB_training_path = os.path.join(root_data_path, "training")

#validation data path
validation_data_path=os.path.join(root_data_path, "validation")

#testing data path
test_data_path=os.path.join(root_data_path, "test")

In [5]:
#model path
models_path = "models"

#results path
results_path = "results"

#evaluation result path
evaluations_path = "evaluations"

os.makedirs(models_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)
os.makedirs(evaluations_path, exist_ok=True)

# Training data

In [6]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(taskB_training_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])

print ("No. of job_id and skill_id triple: {}".format(job2skill.shape[0]))
job2skill.head()

No. of job_id and skill_id triple: 114699


,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [7]:
job2skill['skill_id'][:10]

0    http://data.europa.eu/esco/skill/93a68dcb-3dc6...
1    http://data.europa.eu/esco/skill/05bc7677-5a64...
2    http://data.europa.eu/esco/skill/860be36a-d19b...
3    http://data.europa.eu/esco/skill/fed5b267-73fa...
4    http://data.europa.eu/esco/skill/f64fe2c2-d090...
5    http://data.europa.eu/esco/skill/271a36a0-bc7a...
6    http://data.europa.eu/esco/skill/591dd514-735b...
7    http://data.europa.eu/esco/skill/47ed1d37-971b...
8    http://data.europa.eu/esco/skill/892f8e2f-189a...
9    http://data.europa.eu/esco/skill/a2dfc063-7b24...
Name: skill_id, dtype: object

In [8]:
jobid2terms_path = os.path.join(taskB_training_path, "jobid2terms.json")
with open(jobid2terms_path, "r", encoding="utf-8") as f:
    jobid2terms = json.load(f)

In [9]:
print ("Total job ids: {}".format(len(jobid2terms)))


Total job ids: 3039


In [10]:
skillid2terms_path = os.path.join(taskB_training_path, "skillid2terms.json")
with open(skillid2terms_path, "r", encoding="utf-8") as f:
    skillid2terms = json.load(f)


In [11]:
print ("Total skill ids: {}".format(len(skillid2terms)))


Total skill ids: 13939


In [12]:

job2skill["job_terms"] = job2skill["job_id"].map(jobid2terms)
job2skill["skill_terms"] = job2skill["skill_id"].map(skillid2terms)

In [13]:
job2skill.describe()


,job_id,skill_id,rel_type,job_terms,skill_terms
count,114699,114699,114699,114699,114699
unique,3011,13224,2,3011,13224
top,http://data.europa.eu/esco/occupation/ae8b19cc...,http://data.europa.eu/esco/skill/03b9b491-fc9b...,essential,"[optomechanical engineer, optomechanical techn...","[create solutions to problems, create solution..."
freq,69,328,62480,69,328


In [14]:
job_id_to_search = job2skill.job_id.to_list()[100]
job_data = job2skill[job2skill.job_id == job_id_to_search]

In [15]:
job_data.job_terms.iloc[0]

['air traffic safety technician',
 'air traffic safety electronics hardware specialist',
 'air traffic safety software specialist',
 'air traffic safety engineer',
 'air traffic safety hardware specialist',
 'air traffic safety electronics software specialist',
 'air traffic safety electronics engineer',
 'air traffic safety electronics technician']

In [16]:
job_data.rel_type.value_counts()

rel_type
optional     31
essential    24
Name: count, dtype: int64

In [17]:
job_data.skill_terms.to_list()[0]


['common aviation safety regulations',
 'common safety regulations in aviation',
 'standard regulations in aviation safety',
 'prevailing safety regulations in civil aviation',
 'regulations governing safety in international civil aviation',
 'common safety legislation in aviation',
 'common civil aviation safety regulations',
 'prevailing aviation safety directives',
 'common aviation safety legislation',
 'common international civil aviation safety regulations',
 'standard aviation safety regulations']

In [18]:
job2skill.job_id.to_list()[100]


'http://data.europa.eu/esco/occupation/0022f466-426c-41a4-ac96-a235c945cf97'

# Validation data

In [19]:

validation_queries_path = os.path.join(validation_data_path, "queries")
validation_corpus_elements_path = os.path.join(validation_data_path, "corpus_elements")

In [20]:
validation_queries = pd.read_csv(validation_queries_path, sep="\t")
validation_corpus_elements = pd.read_csv(validation_corpus_elements_path, sep="\t")

In [21]:
print ("Number of queries and copurs elements in validation dataset: {} and {}".format(len(validation_queries), len(validation_corpus_elements)))


Number of queries and copurs elements in validation dataset: 304 and 1439


In [22]:
print ("validation query samples:")
validation_queries.head()

validation query samples:


,q_id,jobtitle
0,dev_qb_jt_1,corporate governance analyst
1,dev_qb_jt_2,logistics business analyst
2,dev_qb_jt_3,operations planning analyst
3,dev_qb_jt_4,real estate data analyst
4,dev_qb_jt_5,security analyst ii


In [23]:
validation_queries.jobtitle[0]


'corporate governance analyst'

In [24]:
print ("validation corpus elements samples:")
validation_corpus_elements.head()

validation corpus elements samples:


,c_id,esco_uri,skill_aliases
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"['pricing plans', 'price strategies', 'pricing..."
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"['putting out fires', 'put out fires', 'exting..."
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"['online assessment', 'analysis of web strateg..."
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"['keeping up with trends', 'keep pace with tre..."
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"['prepare tax returns form', 'complete tax ret..."


In [25]:
validation_corpus_elements["skill_aliases"] = validation_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))


In [26]:
validation_corpus_elements.skill_aliases[0]


['pricing plans',
 'price strategies',
 'pricing tactics',
 'pricing strategies',
 'pricing strategy']

In [27]:
def aggregate_list_of_terms(terms_list):
    '''
        aggregate the list of terms to a long text (string)
    '''
    return " ".join(terms_list)

validation_corpus_elements["skill_document"] = validation_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


In [28]:
validation_corpus_elements.skill_document[0]

'pricing plans price strategies pricing tactics pricing strategies pricing strategy'

In [29]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
validation_queries_ids = validation_queries.q_id.to_list()
validation_queries_texts = validation_queries.jobtitle.to_list()

validation_queries_map = dict(zip(validation_queries_ids, validation_queries_texts))

list(validation_queries_map.items())[0]

('dev_qb_jt_1', 'corporate governance analyst')

In [30]:
validation_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,dev_cb_sk_1,http://data.europa.eu/esco/skill/1c460d2d-90c6...,"[pricing plans, price strategies, pricing tact...",pricing plans price strategies pricing tactics...
1,dev_cb_sk_2,http://data.europa.eu/esco/skill/301a6581-e983...,"[putting out fires, put out fires, extinguish ...",putting out fires put out fires extinguish fir...
2,dev_cb_sk_3,http://data.europa.eu/esco/skill/a4881e54-6055...,"[online assessment, analysis of web strategy, ...",online assessment analysis of web strategy web...
3,dev_cb_sk_4,http://data.europa.eu/esco/skill/efda73b4-5212...,"[keeping up with trends, keep pace with trends...",keeping up with trends keep pace with trends f...
4,dev_cb_sk_5,http://data.europa.eu/esco/skill/22a173f5-868c...,"[prepare tax returns form, complete tax return...",prepare tax returns form complete tax returns ...


In [31]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
'''
validation_documents_ids = validation_corpus_elements.c_id.to_list()
validation_documents_esco_uri = validation_corpus_elements.esco_uri.to_list()
validation_documents_texts = validation_corpus_elements.skill_document.to_list()

validation_documents_map = dict(zip(validation_documents_ids, validation_documents_texts))

list(validation_documents_map.items())[0]

('dev_cb_sk_1',
 'pricing plans price strategies pricing tactics pricing strategies pricing strategy')

In [32]:
len(validation_documents_texts)
validation_documents_texts[0]

'pricing plans price strategies pricing tactics pricing strategies pricing strategy'

In [38]:
from bm25 import BM25

#from sklearn.feature_extraction.text import TfidfVectorizer
#from sklearn.metrics.pairwise import linear_kernel

bm25 = BM25()

bm25.fit(validation_documents_texts)


In [39]:
query_doc_similarity = []
for query_text in validation_queries_texts:
    query_doc_bm25 = bm25.transform(query_text, validation_documents_texts)
    query_doc_similarity.append(query_doc_bm25)
    #X_queries = bm25.transform(validation_queries_texts)

In [43]:
print ("{}, {}".format(len(query_doc_similarity), len(query_doc_similarity[0])))

304, 1439


In [44]:
validation_similarities_query_documents = np.array(query_doc_similarity)

In [45]:
#validation_similarities_query_documents = linear_kernel(X_queries, X_docs)
print("Similarity matrix shape:", validation_similarities_query_documents.shape)

Similarity matrix shape: (304, 1439)


In [46]:
def get_ranked_result_list(queries_ids, queries_map, documents_ids, documents_map, similarities_query_documents, documents_texts, model_name):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities_query_documents[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = documents_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = queries_map[q_id]
            doc_name = documents_texts[c_idx]
            score = similarities_query_documents[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

    return results, results_name

results_list, results_name = get_ranked_result_list(validation_queries_ids, validation_queries_map,
                                                    validation_documents_ids, validation_documents_map,
                                                    validation_similarities_query_documents, validation_documents_texts,
                                                   "Adhoc_ST_Cosine")
results_path

'results'

In [47]:
result_adhoc_taskB = os.path.join(results_path, "validation_adhoc_bm25_kB.trec")
with open(result_adhoc_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list))

In [48]:
qrels_file = os.path.join(validation_data_path, "qrels.tsv")

run_validation_adhoc_file = os.path.join(results_path, "validation_adhoc_bm25_kB.trec")

In [49]:
# run this cell just once for the first time
#!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
#!pip install -r talentclef25_evaluation_script/requirements.txt

In [50]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_validation_adhoc_file]

Evaluation


In [51]:

result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

Received parameters:
  qrels: taskB\validation\qrels.tsv
  run: results\validation_adhoc_bm25_kB.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.0894
mrr: 0.5691
ndcg: 0.5615
precision@5: 0.3263
precision@10: 0.2628
precision@100: 0.0815



In [52]:
print("qrels_file:", qrels_file)
print("run_file:", run_validation_adhoc_file)

qrels_file: taskB\validation\qrels.tsv
run_file: results\validation_adhoc_bm25_kB.trec


# Test data

In [42]:
test_queries_path = os.path.join(test_data_path, "queries")
test_corpus_elements_path = os.path.join(test_data_path, "corpus_elements")

In [43]:
test_queries = pd.read_csv(test_queries_path,sep="\t")
test_corpus_elements = pd.read_csv(test_corpus_elements_path, sep="\t")

In [44]:
print ("Number of queries and copurs elements in test dataset: {} and {}".format(len(test_queries), len(test_corpus_elements)))

Number of queries and copurs elements in test dataset: 1520 and 1986


In [45]:
print ("test query samples:")
test_queries.head()

test query samples:


,q_id,jobtitle
0,15ec5e80,Qdoba Mexican Eats
1,a0d97669,Hairstylist
2,4b3976f7,Digital Logic
3,a3bc11cc,Interstate/Local Moving Contractor
4,e12ee1fd,Press Operator


In [46]:
test_corpus_elements["skill_aliases"] = test_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))


In [47]:
test_corpus_elements.skill_aliases[1]


['sales activities',
 'sales exercises',
 'a sales activity',
 'sales actions',
 'sales activity',
 'selling activities',
 'sales operations']

In [48]:
def aggregate_list_of_terms(terms_list):
    '''
        aggregate the list of terms to a long text (string)
    '''
    return " ".join(terms_list)

test_corpus_elements["skill_document"] = test_corpus_elements["skill_aliases"].apply(lambda x: aggregate_list_of_terms(x))

test_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,[recruiting and hiring],recruiting and hiring
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"[sales activities, sales exercises, a sales ac...",sales activities sales exercises a sales activ...
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"[carry out job analysis, carrying out job anal...",carry out job analysis carrying out job analys...
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"[develop staff, help strengthen staff, build u...",develop staff help strengthen staff build up s...
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"[hire human resources, hire experienced people...",hire human resources hire experienced people m...


In [49]:
test_corpus_elements.skill_document[0]

'recruiting and hiring'

In [50]:
'''
    build a dictionary/mapping between the query_id and text from the query pandas dataframe
'''
test_queries_ids = test_queries.q_id.to_list()
test_queries_texts = test_queries.jobtitle.to_list()

test_queries_map = dict(zip(test_queries_ids, test_queries_texts))

list(test_queries_map.items())[0]

('15ec5e80', 'Qdoba Mexican Eats')

In [51]:
test_corpus_elements.head()

,c_id,esco_uri,skill_aliases,skill_document
0,f3a278e7,http://data.europa.eu/esco/skill/77e41477-4ecf...,[recruiting and hiring],recruiting and hiring
1,2938fcd6,http://data.europa.eu/esco/skill/450de541-797a...,"[sales activities, sales exercises, a sales ac...",sales activities sales exercises a sales activ...
2,7b3c55d4,http://data.europa.eu/esco/skill/0fbaceff-df05...,"[carry out job analysis, carrying out job anal...",carry out job analysis carrying out job analys...
3,572ec5d6,http://data.europa.eu/esco/skill/714eebcb-569a...,"[develop staff, help strengthen staff, build u...",develop staff help strengthen staff build up s...
4,73f0560b,http://data.europa.eu/esco/skill/5aa57bdc-5765...,"[hire human resources, hire experienced people...",hire human resources hire experienced people m...


In [52]:
'''
    build a dictionary/mapping between the corpus_id and document from the pandas dataframe
'''
test_documents_ids = test_corpus_elements.c_id.to_list()
test_documents_esco_uri = test_corpus_elements.esco_uri.to_list()
test_documents_texts = test_corpus_elements.skill_document.to_list()

test_documents_map = dict(zip(test_documents_ids, test_documents_texts))

list(test_documents_map.items())[0]

('f3a278e7', 'recruiting and hiring')

In [53]:
len(test_documents_texts)
test_documents_texts[0]

'recruiting and hiring'

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),  
    min_df=1,
    max_features=5000,
    norm="l2"
)

X_docs = tfidf.fit_transform(test_documents_texts)
X_queries = tfidf.transform(test_queries_texts)

In [55]:
print (X_docs[0][:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 3749)	0.6842758425605336
  (0, 206)	0.29607070137149183
  (0, 1949)	0.6664148190710233


In [56]:
X_docs.shape, X_queries.shape

((1986, 5000), (1520, 5000))

In [57]:
test_similarities_query_documents = linear_kernel(X_queries, X_docs)
print("Similarity matrix shape:", test_similarities_query_documents.shape)

Similarity matrix shape: (1520, 1986)


In [58]:
print (test_similarities_query_documents[0,0])

0.0


In [59]:
def get_ranked_result_list(queries_ids, queries_map, documents_ids, documents_map, similarities_query_documents, documents_texts, model_name):
    results = []
    results_name = []
    
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities_query_documents[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  
            doc_id = documents_ids[c_idx]
            
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = queries_map[q_id]
            doc_name = documents_texts[c_idx]
            score = similarities_query_documents[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

    return results, results_name

results_list, results_name = get_ranked_result_list(test_queries_ids, test_queries_map,
                                                    test_documents_ids, test_documents_map,
                                                    test_similarities_query_documents, test_documents_texts,
                                                   "Adhoc_ST_Cosine")
results_path

'results'

In [60]:
result_adhoc_taskB = os.path.join(results_path, "test_adhoc_tfidf_kB.trec")
with open(result_adhoc_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_list))

In [61]:
qrels_file = os.path.join(test_data_path, "qrels.tsv")

run_test_adhoc_file = os.path.join(results_path, "test_adhoc_tfidf_kB.trec")

In [62]:
print ("Evaluation")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_test_adhoc_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

Evaluation
Received parameters:
  qrels: taskB\test\qrels.tsv
  run: results\test_adhoc_tfidf_kB.trec
Loading qrels...



In [63]:
print("qrels_file:", qrels_file)
print("run_file:", run_test_adhoc_file)

qrels_file: taskB\test\qrels.tsv
run_file: results\test_adhoc_tfidf_kB.trec
